<a href="https://colab.research.google.com/github/Pgiuliano8/COMS5790-NLP-Projects/blob/main/Project_3/project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **COMS 5790 - Project 3: RAG System for QA Task**

In [1]:
# pandas and sentence-transformers are already installed on Colab (no -U, so Colab's packages are not broken)
!pip install -q langchain-community langchain-text-splitters langchain-huggingface faiss-cpu rank_bm25 rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1. Data Loading & Text Pre-processing
In this section, I load the dataset and clean the raw lecture transcripts to create a unified and coherent text block.

In [2]:
import re
import pandas as pd

qa_test = pd.read_csv("QA_test.csv")
qa_test["Questions"] = qa_test["Questions"].str.strip()
print(f"Loaded {len(qa_test)} questions")

with open("documents.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Newlines and empty lines are only pauses of the speaker, not paragraphs: join everything into one text
clean_text = re.sub(r"\s+", " ", raw_text).strip()
print(f"Original text length: {len(raw_text)} characters | cleaned: {len(clean_text)} characters")
print("\nPreview of the cleaned text:\n" + clean_text[:200] + "...")

Loaded 110 questions
Original text length: 77504 characters | cleaned: 76034 characters

Preview of the cleaned text:
Welcome to this lecture on animal ecology. You will learn something about toucans today. You’ve probably seen a toucan before. Maybe on a cereal box, maybe at a zoo, or in a nature documentary. Toucan...


### The 5 known answers
The instructor gave the correct answers to questions 1-5. I use them with their **original wording** as few-shot examples,
and as a small validation set (ROUGE-L, the Kaggle metric).

I also detect the **yes/no questions** (they start with an auxiliary verb, e.g. *Was*, *Can*, *Is this true?*): only these questions
get the "answer Yes or No" rule. Giving this rule to every question made Flan-T5-XL answer Yes/No almost everywhere in a previous run.

In [3]:
KNOWN_ANSWERS = {1: "Representative", 2: "Arithmetic logic unit", 3: "int", 4: "1993", 5: "Green laser"}
FEW_SHOT = [(qid, qa_test.loc[qa_test.ID == qid, "Questions"].item(), ans) for qid, ans in KNOWN_ANSWERS.items()]


def is_yes_no(question):
    q = question.lower()
    return bool(re.match(r"^(is|are|was|were|can|could|does|do|did|has|have|should|will|would)\b", q)) or "true?" in q


print("Yes/no questions:", [qid for qid, q in zip(qa_test.ID, qa_test.Questions) if is_yes_no(q)])

Yes/no questions: [20, 55, 61]


## 2. Vector Database & Retrieval
I split the cleaned text into chunks of 400 characters with an overlap of 50 characters, and store their embeddings in a FAISS vector database.

Two retrieval modes are compared:
- **dense** (first version): the 3 chunks whose embedding is closest to the question (same *meaning*);
- **hybrid**: the same 3 chunks plus the 2 best chunks according to **BM25** (same *words*, useful for names, dates and exact terms).

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = text_splitter.split_text(clean_text)
print(f"Total chunks created: {len(chunks)}")

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_texts(chunks, embedding_model)
dense_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

sparse_retriever = BM25Retriever.from_texts(chunks)
sparse_retriever.k = 2


def retrieve(question, mode):
    docs = dense_retriever.invoke(question)
    if mode == "hybrid":
        docs += sparse_retriever.invoke(question)
    unique = []
    for d in docs:                      # union without duplicates, dense chunks first
        if d.page_content not in unique:
            unique.append(d.page_content)
    return unique


# Test with a real question (not with the answer word)
test_question = FEW_SHOT[0][1]
print(f"\nTest question: {test_question}")
for i, c in enumerate(retrieve(test_question, "hybrid"), 1):
    print(f"[{i}] {c}")

/tmp/ipykernel_470/3560398066.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Total chunks created: 224


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Test question: A sample is unbiased if it is ___________ of the population.
[1] . The key here is that your sample needs to be unbiased. That’s an important word to know, unbiased. If you’re not unbiased, then what are you? Biased. You’re biased if your sample tends to favor one group over another. For example, you want to know who people think is going to win the superbowl
[2] .” What makes a good sample? What should the sample look like? Well, it should look like the population! This sounds simple, but getting a sample to look like the population can be tricky. We will talk about this later. Now, there are two types of statistics, and they are called descriptive statistics and inferential statistics
[3] . And a sample is simply a subset of the population. We say the population is too difficult, or too expensive, or too unwieldy to work with. If you can’t get a whole population, then your best bet is to work with a sample. In this class, we’ll use the big N to refer to the population

## 3. LLM & Prompt Engineering
I use **Flan-T5**, an instruction-tuned seq2seq model that is good at short, extractive answers.

Two prompts are compared:
- **original**: my first prompt (rules for every question + 5 paraphrased examples);
- **improved**: the same rules, but (i) the 5 examples are the real questions with their correct answers, (ii) the yes/no rule is given
  only to yes/no questions, (iii) for questions 1-5 the example of the same question is removed, so the model cannot copy its answer.

Flan-T5 was trained with inputs of about 512 tokens, so with the improved prompt the last retrieved chunk is dropped when the prompt is
too long (the question is never cut).

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MAX_INPUT_TOKENS = 600
MAX_NEW_TOKENS = 30
tokenizer, model, loaded_name = None, None, None


def load_model(name):
    global tokenizer, model, loaded_name
    if name == loaded_name:
        return
    model = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(name)
    # float32: T5 models can overflow in float16
    model = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=torch.float32, device_map="auto")
    model.eval()
    loaded_name = name
    print(f"Loaded {name}")


ORIGINAL_EXAMPLES = '''Question: What represents the entire collection of objects?
Answer: Representative

Question: What does ALU stand for?
Answer: Arithmetic logic unit

Question: What represents integers in Python?
Answer: int

Question: When was the Wild Bird Conservation Act passed?
Answer: 1993

Question: What kind of laser is used?
Answer: Green laser'''


def build_prompt(context, question, qid, style):
    if style == "original":
        examples = ORIGINAL_EXAMPLES
        yes_no_rule = '- For Yes/No questions, answer ONLY "Yes" or "No".\n'
    else:
        examples = "\n\n".join(f"Question: {q}\nAnswer: {a}" for i, q, a in FEW_SHOT if i != qid)
        yes_no_rule = '- This is a Yes/No question: answer ONLY "Yes" or "No".\n' if is_yes_no(question) else ""
    return f'''You are a highly precise question-answering assistant.
Answer the question based EXCLUSIVELY on the provided context. Ignore any real-world factual knowledge.

Strict Output Rules:
- Keep the answer as short and precise as possible (e.g., output "Cloudy" instead of "The weather tomorrow is cloudy").
- Do not use abbreviations.
{yes_no_rule}- For fill-in-the-blank questions containing "___________", output ONLY the missing exact words.
- If the context does not contain enough information to answer, output EXACTLY "I don't know".

Examples of the exact expected output format:
{examples}

Now answer the following:

Context:
{context}

Question: {question}
Answer:'''


def clean_answer(text, question):
    answer = text.strip().split("\n")[0].strip()
    answer = re.sub(r"^(A|Answer)\s*:\s*", "", answer).strip().strip('."\'').strip()
    if answer == "":
        return "I don't know"
    if is_yes_no(question) and re.match(r"^(yes|no)\b", answer.lower()):
        return answer.split()[0].strip(",.!").capitalize()
    return answer


@torch.no_grad()
def answer_question(question, qid, style, mode, clean):
    context_chunks = retrieve(question, mode)
    prompt = build_prompt(" ".join(context_chunks), question, qid, style)
    if style == "improved":
        while len(tokenizer(prompt).input_ids) > MAX_INPUT_TOKENS and len(context_chunks) > 1:
            context_chunks = context_chunks[:-1]           # drop the last chunk, never the question
            prompt = build_prompt(" ".join(context_chunks), question, qid, style)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)     # greedy decoding
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_answer(text, question) if clean else text.strip()

## 4. Experiments
Each variant changes **one thing** with respect to the previous one, so the effect of each change can be measured:

| Variant | Model | Prompt | Retrieval | Answer cleaning |
|---|---|---|---|---|
| A_original | flan-t5-large | original | dense | no |
| B_prompt | flan-t5-large | improved | dense | yes |
| C_xl | flan-t5-xl | improved | dense | yes |
| D_hybrid | flan-t5-xl | improved | hybrid | yes |

For each variant I save a Kaggle file and print: ROUGE-L F1 on the 5 known answers, the number of "Yes"/"No" answers given to questions
that are **not** yes/no questions (should be 0), the number of "I don't know" and the mean length of the answers.

In [9]:
from rouge_score import rouge_scorer
from tqdm.notebook import tqdm

scorer = rouge_scorer.RougeScorer(["rougeL"])

VARIANTS = [
    # name,          model,                 prompt,     retrieval, cleaning
    ("A_original", "google/flan-t5-large", "original", "dense", False),
    ("B_prompt",   "google/flan-t5-large", "improved", "dense", True),
    ("C_xl",       "google/flan-t5-xl",    "improved", "dense", True),
    ("D_hybrid",   "google/flan-t5-xl",    "improved", "hybrid", True),
]

summary = []
for name, model_name, style, mode, clean in VARIANTS:
    load_model(model_name)
    answers = [answer_question(q, qid, style, mode, clean)
               for qid, q in tqdm(zip(qa_test.ID, qa_test.Questions), total=len(qa_test), desc=name)]
    sub = pd.DataFrame({"ID": qa_test.ID, "answer": answers})
    sub.to_csv(f"submission_{name}.csv", index=False)          # Kaggle header: ID,ANSWER

    known = [scorer.score(KNOWN_ANSWERS[i], sub.loc[sub.ID == i, "answer"].item())["rougeL"].fmeasure for i in KNOWN_ANSWERS]
    wrong_yes_no = sum(a.strip(".").lower() in ("yes", "no") and not is_yes_no(q) for a, q in zip(answers, qa_test.Questions))
    summary.append({"variant": name, "ROUGE-L (5 known)": round(sum(known) / 5, 2), "Yes/No to non yes/no questions": wrong_yes_no,
                    "I don't know": answers.count("I don't know"), "mean words": round(sum(len(a.split()) for a in answers) / len(answers), 1)})
    print(summary[-1])

pd.DataFrame(summary)

A_original:   0%|          | 0/110 [00:00<?, ?it/s]

{'variant': 'A_original', 'ROUGE-L (5 known)': 0.4, 'Yes/No to non yes/no questions': 4, "I don't know": 19, 'mean words': 3.2}


B_prompt:   0%|          | 0/110 [00:00<?, ?it/s]

{'variant': 'B_prompt', 'ROUGE-L (5 known)': 0.2, 'Yes/No to non yes/no questions': 0, "I don't know": 19, 'mean words': 3.2}


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

...

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (515 > 512). Running this sequence through the model will result in indexing errors


{'variant': 'C_xl', 'ROUGE-L (5 known)': 0.4, 'Yes/No to non yes/no questions': 0, "I don't know": 13, 'mean words': 3.6}


D_hybrid:   0%|          | 0/110 [00:00<?, ?it/s]

{'variant': 'D_hybrid', 'ROUGE-L (5 known)': 0.4, 'Yes/No to non yes/no questions': 0, "I don't know": 13, 'mean words': 3.7}


,variant,ROUGE-L (5 known),Yes/No to non yes/no questions,I don't know,mean words
0,A_original,0.4,4,19,3.2
1,B_prompt,0.2,0,19,3.2
2,C_xl,0.4,0,13,3.6
3,D_hybrid,0.4,0,13,3.7


### Answers that change between variants

In [12]:
table = pd.DataFrame({"ID": qa_test.ID, "question": qa_test.Questions})
for name, *_ in VARIANTS:
    table[name] = pd.read_csv(f"submission_{name}.csv")["answer"]
names = [v[0] for v in VARIANTS]
table[table[names].nunique(axis=1) > 1].head(40)

,ID,question,A_original,B_prompt,C_xl,D_hybrid
0,1,A sample is unbiased if it is ___________ of t...,subset,subset,I don't know,I don't know
1,2,What is the brain of the computer?,ALU,ALU,arithmetic logic unit,arithmetic logic unit
2,3,What term does Python use to represent integers?,int,I-N-T,I-N-T,I-N-T
4,5,What type of light would a laser with an emera...,No,I don't know,I don't know,I don't know
7,8,Why does a computer use the binary system inst...,Because a transistor deals with only two state...,I don't know,Because a transistor deals with only two state...,Because a transistor deals with only two state...
9,10,"In programming, what type of number is 2.33 or...",I don't know,I don't know,Rational point,Rational point
12,13,"When visiting a tropical locale, you're more l...",Yes,Toucans are rainforest dwellers. You can usual...,Their large beaks makes it hard to balance,Their large beaks makes it hard to balance
13,14,Where do toucans sleep?,in such a small and enclosed space like a tree...,in such a small and enclosed space like a tree...,tree holes,tree holes
15,16,What type of study is done on data that have a...,retrospective,retrospective,Retrospective,Retrospective study
17,18,Give one example of data collection that invol...,Survey,Survey,A survey,A survey
